# Role Assignment Coherence Judge

This notebook compares two role-assignment artifacts with a language-model-as-a-judge workflow.
It relies on `RoleAssignmentJudge` (OPENAI judge by default) that was implemented in `src/role_evaluator`.

## 1. Configuration
- Set a gpt API key (best judged model) via the `OPENAI_API_KEY` environment variable.
- Update the file paths below to point at the two role-assignment JSON dumps you want to compare.

In [1]:
import os
import sys
from pathlib import Path
import json
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve().parent.parent  # Go up two levels to reach project root
sys.path.append(str(PROJECT_ROOT))  # Add project root to Python path

from src.role_evaluator import RoleAssignmentJudge, DatasetComparisonSpec

# Role assignment file paths
SPIDER_ASSIGNMENT_NAIVE = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_spider_dev_20251005_145223.json'
SPIDER_ASSIGNMENT_WITH_COT = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_spider_dev_20250930_230436.json'
BIRD_ASSIGNMENT_NAIVE = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_bird_dev_20251005_164218.json'
BIRD_ASSIGNMENT_WITH_COT = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_bird_dev_selected.json'
LIVESQL_ASSIGNMENT_NAIVE = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_20251005_164446_livesql_dev.json'
LIVESQL_ASSIGNMENT_WITH_COT = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_20251001_142136_livesql_dev.json'

# ============================================================================
# JUDGE MODEL CONFIGURATION
# ============================================================================
# Choose one of the following options:

# Option 1: OpenAI GPT-4o (Recommended for best quality)
JUDGE_MODEL = 'gpt-5'
JUDGE_API_KEY_ENV = 'OPENAI_API_KEY'

# Option 2: Google Gemini 2.5 Flash (Fast and cost-effective)
# JUDGE_MODEL = 'gemini-2.5-flash'
# JUDGE_API_KEY_ENV = 'GEMINI_API_KEY'

# ============================================================================

# Read API key from environment
API_KEY = os.environ.get(JUDGE_API_KEY_ENV, '').strip()
if not API_KEY:
    raise ValueError(f'Set {JUDGE_API_KEY_ENV} in your environment or assign API_KEY manually.')

print(f"Using judge model: {JUDGE_MODEL}")
print(f"API key source: {JUDGE_API_KEY_ENV}")

# Dataset comparison specifications
DATASET_COMPARISONS = [
    DatasetComparisonSpec(
        label='spider',
        assignment_a_path=SPIDER_ASSIGNMENT_NAIVE,
        assignment_b_path=SPIDER_ASSIGNMENT_WITH_COT,
    ),
    DatasetComparisonSpec(
        label='bird',
        assignment_a_path=BIRD_ASSIGNMENT_NAIVE,
        assignment_b_path=BIRD_ASSIGNMENT_WITH_COT,
    ),
    DatasetComparisonSpec(
        label='livesql',
        assignment_a_path=LIVESQL_ASSIGNMENT_NAIVE,
        assignment_b_path=LIVESQL_ASSIGNMENT_WITH_COT,
    ),
]

/home/feiy/anaconda3/envs/llm4db/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using judge model: gpt-5
API key source: OPENAI_API_KEY


### Quick Setup Examples

**Using OpenAI GPT-4o:**
```bash
export OPENAI_API_KEY="sk-..."
```
Set: `JUDGE_MODEL = 'gpt-4o'` and `JUDGE_API_KEY_ENV = 'OPENAI_API_KEY'`

**Using Google Gemini 2.5 Flash (Recommended for cost):**
```bash
export GEMINI_API_KEY="AIza..."
```
Set: `JUDGE_MODEL = 'gemini-2.5-flash'` and `JUDGE_API_KEY_ENV = 'GEMINI_API_KEY'`

**Using Google Gemini 2.0 Flash:**
```bash
export GEMINI_API_KEY="AIza..."
```
Set: `JUDGE_MODEL = 'gemini-2.0-flash-exp'` and `JUDGE_API_KEY_ENV = 'GEMINI_API_KEY'`

Get your Gemini API key at: https://aistudio.google.com/app/apikey

## 2. Instantiate the judge

The judge model is configured in the previous cell. Supported models include:

**OpenAI Models:**
- `gpt-5` - Latest GPT model (⚠️ **Note**: Uses fixed temperature=1.0)
- `gpt-4o` - Best quality with customizable parameters
- `gpt-4o-mini` - Fast and economical
- `gpt-4-turbo` - Previous generation

**Google Gemini Models:**
- `gemini-2.5-flash` - Latest Flash model ⭐ **Recommended for cost-effectiveness**
- `gemini-2.0-flash-exp` - Experimental 2.0 Flash
- `gemini-1.5-pro` - High quality reasoning
- `gemini-1.5-flash` - Balanced speed/quality

The judge instance is initialized with appropriate temperature and token limits for evaluation tasks.

**Special Note**: GPT-5 and o1-series models have fixed temperature (1.0) and may not support top_p parameter. The code automatically adjusts these settings.

### Model-Specific Parameter Constraints

Different models have different parameter requirements:

| Model | Temperature | top_p | Notes |
|-------|------------|-------|-------|
| `gpt-5` | ⚠️ Fixed at 1.0 | ❌ Not supported | Auto-adjusted by code |
| `gpt-5-mini` | ⚠️ Fixed at 1.0 | ❌ Not supported | Auto-adjusted by code |
| `o1`, `o1-mini` | ⚠️ Fixed at 1.0 | ❌ Not supported | Auto-adjusted by code |
| `gpt-4o` | ✅ Customizable | ✅ Supported | Full control |
| `gemini-*` | ✅ Customizable | ✅ Supported | Full control |

The `RoleAssignmentJudge` class automatically handles these constraints, so you don't need to worry about them when configuring.

In [2]:
# Initialize the judge with configured model and API key
judge = RoleAssignmentJudge(
    model=JUDGE_MODEL,
    api_key=API_KEY,
    temperature=0.9,
    top_p=0.9,
    max_completion_tokens=800,
)

print(f"Judge initialized successfully with model: {JUDGE_MODEL}")
judge

Judge initialized successfully with model: gpt-5


### Quick Test (Optional)

Before running the full evaluation, you can test if the judge is working properly:

In [3]:
# Quick test query
# Use the judge's configured temperature (already adjusted for GPT-5 if needed)
test_response = judge.oracle.query(
    prompt_sys="You are an expert database evaluator.",
    prompt_user="In one sentence, what makes a good database role assignment?",
    temp=judge.temperature,  # Use judge's temperature (1.0 for GPT-5, 0.9 for others)
    top_p=judge.top_p,
    max_completion_tokens=100,
)

print("Test Response:")
print(test_response.get('answer', 'No answer received'))

Test Response:



## 3. Run evaluation
This triggers one OPENAI call per overlapping database in the two artifacts. Optional filters:
- `databases=[...]`: restrict to a selected list.
- `limit=N`: judge only the first *N* shared databases (useful for pilots).

In [ ]:
batch_result = judge.evaluate_many(
    DATASET_COMPARISONS,
    combined_progress=True,
    dataset_progress=True,
    progress_description='Evaluating all databases',
)
batch_result.summary.to_dict()

Evaluating all databases:  25%|██▌       | 11/44 [05:02<14:57, 27.18s/db]

In [ ]:
per_dataset_summary = {
    label: per_result.summary.to_dict()
    for label, per_result in batch_result.results.items()
}
per_dataset_summary

In [ ]:
result_all = {
    'overall': batch_result.summary.to_dict(),
    'per_dataset': per_dataset_summary,
}
result_all

## 4. Inspect detailed decisions
Each row corresponds to one database comparison.

In [ ]:
import pandas as pd

decision_records = []
for label, per_result in batch_result.results.items():
    for decision in per_result.decisions:
        record = decision.to_dict()
        record['dataset'] = label
        decision_records.append(record)

decisions_df = pd.DataFrame(decision_records)
decisions_df

## 5. Persist the raw outputs (optional)
Use this to archive the JSON that records both per-db judgements and aggregate statistics.

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = Path('outputs/judge_results') / f'coherence_judge_{timestamp}.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open('w', encoding='utf-8') as handle:
    json.dump(batch_result.to_dict(), handle, indent=2)
output_path